In [1]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.text import tokenizer_from_json
import json
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
model = load_model("model/summarizer_model.keras")

C:\Users\Pranjal\anaconda3\Lib\site-packages\keras\src\saving\saving_lib.py:801: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 12 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [8]:
from tensorflow.keras.preprocessing.text import tokenizer_from_json

with open("model/tokenizer.json") as f:
    tokenizer = tokenizer_from_json(f.read())

In [10]:
def predict_summary(text):
    seq = tokenizer.texts_to_sequences([text])
    seq = pad_sequences(seq, maxlen=100, padding='post')

    # Start token (must exist in your tokenizer)
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = tokenizer.word_index.get('start', 1)

    summary = []

    for _ in range(19):  # max summary length
        pred = model.predict([seq, target_seq], verbose=0)

        # get last predicted word
        next_word_id = np.argmax(pred[0, -1, :])
        next_word = tokenizer.index_word.get(next_word_id, '')

        if next_word == 'end' or next_word == '':
            break

        summary.append(next_word)

        # append predicted word to input
        target_seq = np.append(target_seq, [[next_word_id]], axis=1)

    return ' '.join(summary)